In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)


In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)


In [3]:
# ==== CONFIG: EDIT THIS PART ====

# Path to your Kaggle dataset CSV file
DATA_PATH = "Product_Sales_Dataset_2023_2024.csv"  # change to your actual file name

# Column names in the raw data (adjust to real names)
DATE_COL       = "Order Date"      # or "Date"
PRODUCT_COL    = "Product Name"    # or "Product", "Product_ID"
SALES_COL      = "Sales"           # or "Total_Sales", "Revenue"
REGION_COL     = "Region"          # or "Country", "State", etc.
CATEGORY_COL   = "Category"        # or "Product_Category", etc.

# Granularity assumptions:
# If your data is at the ORDER level (multiple rows per product per day),
# we will aggregate to daily product-level sales.
AGGREGATE_TO_DAILY = True

# Forecast horizon (in days) beyond the last date in the dataset
FORECAST_HORIZON_DAYS = 30

# ================================


In [5]:
df_raw = pd.read_csv("/content/product_sales_dataset_final.csv")
print("Shape:", df_raw.shape)
df_raw.head()


Shape: (200000, 14)


,Order_ID,Order_Date,Customer_Name,City,State,Region,Country,Category,Sub_Category,Product_Name,Quantity,Unit_Price,Revenue,Profit
0,1,08-23-23,Bianca Brown,Jackson,Mississippi,South,United States,Accessories,Small Electronics,Phone Case,3,201.01,603.03,221.49
1,2,12-20-24,Jared Edwards,Grand Rapids,Michigan,Centre,United States,Accessories,Small Electronics,Charging Cable,4,74.30,297.20,97.09
2,3,01-29-24,Susan Valdez,Minneapolis,Minnesota,Centre,United States,Clothing & Apparel,Sportswear,Nike Air Force 1,1,68.19,68.19,25.47
3,4,11-29-24,Tina Williams,Tallahassee,Florida,South,United States,Clothing & Apparel,Sportswear,Adidas Tracksuit,3,209.64,628.92,231.38
4,5,09-21-23,Catherine Gordon,Baltimore,Maryland,East,United States,Accessories,Bags,Backpack,1,216.63,216.63,42.46


In [6]:
df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Order_ID       200000 non-null  int64  
 1   Order_Date     200000 non-null  object 
 2   Customer_Name  200000 non-null  object 
 3   City           200000 non-null  object 
 4   State          200000 non-null  object 
 5   Region         200000 non-null  object 
 6   Country        200000 non-null  object 
 7   Category       200000 non-null  object 
 8   Sub_Category   200000 non-null  object 
 9   Product_Name   200000 non-null  object 
 10  Quantity       200000 non-null  int64  
 11   Unit_Price    200000 non-null  float64
 12   Revenue       200000 non-null  float64
 13   Profit        200000 non-null  float64
dtypes: float64(3), int64(2), object(9)
memory usage: 21.4+ MB


In [8]:
# ====== STEP 5: Aggregate to Daily Product Sales ======

df = df_raw.copy()

# Clean column names (optional but recommended)
df.columns = df.columns.str.strip()   # removes spaces like " Revenue "
df.rename(columns={"Unit_Price": "Unit_Price"}, inplace=True)

DATE_COL       = "Order_Date"
PRODUCT_COL    = "Product_Name"
SALES_COL      = "Revenue"
REGION_COL     = "Region"
CATEGORY_COL   = "Category"

# Convert to datetime
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

# Aggregate to daily product-level sales
group_cols = [DATE_COL, PRODUCT_COL]

# Your dataset has Region + Category → include them
group_cols.append(REGION_COL)
group_cols.append(CATEGORY_COL)

df_daily = (
    df.groupby(group_cols, as_index=False)[SALES_COL].sum()
)

df_daily = df_daily.sort_values([PRODUCT_COL, DATE_COL]).reset_index(drop=True)

df_daily.head()


,Order_Date,Product_Name,Region,Category,Revenue
0,2023-01-01,Adidas Tracksuit,Centre,Clothing & Apparel,1618.31
1,2023-01-01,Adidas Tracksuit,East,Clothing & Apparel,224.82
2,2023-01-01,Adidas Tracksuit,West,Clothing & Apparel,1020.36
3,2023-01-02,Adidas Tracksuit,Centre,Clothing & Apparel,198.41
4,2023-01-02,Adidas Tracksuit,South,Clothing & Apparel,846.06


In [9]:
# ====== STEP 6: Date Features + Festive Flag ======

df_feat = df_daily.copy()

# Date features
df_feat["year"] = df_feat[DATE_COL].dt.year
df_feat["month"] = df_feat[DATE_COL].dt.month
df_feat["day"] = df_feat[DATE_COL].dt.day
df_feat["dayofweek"] = df_feat[DATE_COL].dt.dayofweek  # Monday=0
df_feat["weekofyear"] = df_feat[DATE_COL].dt.isocalendar().week.astype(int)
df_feat["is_weekend"] = df_feat["dayofweek"].isin([5, 6]).astype(int)

# ---- Festive / Holiday Calendar (example for 2023–2024, adjust if needed) ----
holiday_dates = [
    # 2023
    "2023-11-23",  # Thanksgiving (US)
    "2023-11-24",  # Black Friday
    "2023-12-24", "2023-12-25",  # Christmas Eve & Day
    "2023-12-31", "2024-01-01",  # New Year Eve & Day

    # 2024
    "2024-11-28",  # Thanksgiving
    "2024-11-29",  # Black Friday
    "2024-12-24", "2024-12-25",
    "2024-12-31", "2025-01-01",
]

holiday_dates = pd.to_datetime(holiday_dates)
holiday_window_days = 2  # treat +/- 2 days around each as "festive"

def mark_festive(date_series, holiday_dates, window=2):
    festive = pd.Series(0, index=date_series.index)
    for h in holiday_dates:
        mask = (date_series >= (h - pd.Timedelta(days=window))) & \
               (date_series <= (h + pd.Timedelta(days=window)))
        festive[mask] = 1
    return festive

df_feat["is_festive"] = mark_festive(df_feat[DATE_COL], holiday_dates, window=holiday_window_days)

df_feat.head()


,Order_Date,Product_Name,Region,Category,Revenue,year,month,day,dayofweek,weekofyear,is_weekend,is_festive
0,2023-01-01,Adidas Tracksuit,Centre,Clothing & Apparel,1618.31,2023,1,1,6,52,1,0
1,2023-01-01,Adidas Tracksuit,East,Clothing & Apparel,224.82,2023,1,1,6,52,1,0
2,2023-01-01,Adidas Tracksuit,West,Clothing & Apparel,1020.36,2023,1,1,6,52,1,0
3,2023-01-02,Adidas Tracksuit,Centre,Clothing & Apparel,198.41,2023,1,2,0,1,0,0
4,2023-01-02,Adidas Tracksuit,South,Clothing & Apparel,846.06,2023,1,2,0,1,0,0


In [12]:
# ====== STEP 7: Lag & Rolling Features (FIXED) ======

df_lagged = df_feat.copy()

# Group by product + region + category for time-series lags
group_cols = [PRODUCT_COL, REGION_COL, CATEGORY_COL]

# Sort for safety
df_lagged = df_lagged.sort_values(group_cols + [DATE_COL])

# Lags
lags = [7, 14, 28]
for l in lags:
    df_lagged[f"lag_{l}"] = (
        df_lagged
        .groupby(group_cols)[SALES_COL]
        .shift(l)
    )

# Rolling means on past sales (using transform to keep index aligned)
rolling_windows = [7, 14, 28]
for w in rolling_windows:
    df_lagged[f"roll_mean_{w}"] = (
        df_lagged
        .groupby(group_cols)[SALES_COL]
        .transform(
            lambda s: s.shift(1).rolling(window=w, min_periods=1).mean()
        )
    )

# Drop rows where the largest lag is not available (optional but recommended)
min_lag = max(lags)
df_lagged = df_lagged[df_lagged[f"lag_{min_lag}"].notnull()].reset_index(drop=True)

df_lagged.head()


,Order_Date,Product_Name,Region,Category,Revenue,year,month,day,dayofweek,weekofyear,is_weekend,is_festive,lag_7,lag_14,lag_28,roll_mean_7,roll_mean_14,roll_mean_28
0,2023-02-07,Adidas Tracksuit,Centre,Clothing & Apparel,370.88,2023,2,7,1,6,0,0,1302.48,122.04,1618.31,1394.098571,1262.152143,1004.487500
1,2023-02-10,Adidas Tracksuit,Centre,Clothing & Apparel,123.81,2023,2,10,4,6,0,0,975.60,317.88,198.41,1261.012857,1279.926429,959.936429
2,2023-02-11,Adidas Tracksuit,Centre,Clothing & Apparel,264.56,2023,2,11,5,6,1,0,626.34,2622.01,597.56,1139.328571,1266.064286,957.272143
3,2023-02-13,Adidas Tracksuit,Centre,Clothing & Apparel,236.61,2023,2,13,0,7,0,0,867.84,2629.52,317.64,1087.645714,1097.675000,945.379286
4,2023-02-14,Adidas Tracksuit,Centre,Clothing & Apparel,378.10,2023,2,14,1,7,0,0,1497.86,641.31,327.14,997.470000,926.752857,942.485357


In [13]:
# ====== STEP 8: Train/Test Split (Time-Based) ======

df_lagged = df_lagged.sort_values(DATE_COL).reset_index(drop=True)

max_date = df_lagged[DATE_COL].max()
cutoff_date = max_date - pd.Timedelta(days=90)  # last 90 days as test

print("Max date in data:", max_date)
print("Train/Test cutoff:", cutoff_date)

train_df = df_lagged[df_lagged[DATE_COL] <= cutoff_date].copy()
test_df  = df_lagged[df_lagged[DATE_COL] >  cutoff_date].copy()

print("Train size:", train_df.shape)
print("Test size:", test_df.shape)


Max date in data: 2024-12-31 00:00:00
Train/Test cutoff: 2024-10-02 00:00:00
Train size: (75787, 18)
Test size: (15646, 18)


In [14]:
# ====== STEP 9: Define Features & Target ======

target_col = SALES_COL

numeric_features = [
    "year", "month", "day", "dayofweek", "weekofyear", "is_weekend",
    "is_festive",
] + [f"lag_{l}" for l in lags] + [f"roll_mean_{w}" for w in rolling_windows]

categorical_features = [PRODUCT_COL, REGION_COL, CATEGORY_COL]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

X_train = train_df[numeric_features + categorical_features]
y_train = train_df[target_col]

X_test = test_df[numeric_features + categorical_features]
y_test = test_df[target_col]


Numeric features: ['year', 'month', 'day', 'dayofweek', 'weekofyear', 'is_weekend', 'is_festive', 'lag_7', 'lag_14', 'lag_28', 'roll_mean_7', 'roll_mean_14', 'roll_mean_28']
Categorical features: ['Product_Name', 'Region', 'Category']


In [15]:
# ====== STEP 10: Preprocessing + XGBoost Pipeline ======

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb

# One-hot encode categoricals, pass through numerics
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features),
    ]
)

xgb_reg = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", xgb_reg),
])

model


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Product_Name', 'Region',
                                                   'Category']),
                                                 ('num', 'passthrough',
                                                  ['year', 'month', 'day',
                                                   'dayofweek', 'weekofyear',
                                                   'is_weekend', 'is_festive',
                                                   'lag_7', 'lag_14', 'lag_28',
                                                   'roll_mean_7',
                                                   'roll_mean_14',
                                                   'roll_mean_28'])])),
                ('regressor',
                 XGBRegressor(base_sc...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=6, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=500, n_jobs=-1,
                              num_parallel_tree=None, ...))])

In [16]:
# ====== STEP 11: Train Model ======
%%time
model.fit(X_train, y_train)


CPU times: user 9.44 s, sys: 83.8 ms, total: 9.53 s
Wall time: 6.05 s


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Product_Name', 'Region',
                                                   'Category']),
                                                 ('num', 'passthrough',
                                                  ['year', 'month', 'day',
                                                   'dayofweek', 'weekofyear',
                                                   'is_weekend', 'is_festive',
                                                   'lag_7', 'lag_14', 'lag_28',
                                                   'roll_mean_7',
                                                   'roll_mean_14',
                                                   'roll_mean_28'])])),
                ('regressor',
                 XGBRegressor(base_sc...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=6, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=500, n_jobs=-1,
                              num_parallel_tree=None, ...))])